In [1]:
# Add realistic substructure

# chromatin texture in nucleus

# cristae-like internal mito texture

# ER/Golgi tubular networks

# ribosome-like granularity

# sparse dense granules / nanoparticles

# Add acquisition realism

# Poisson counting noise

# detector blur / PSF

# angular jitter

# missing wedge

# ring artifacts

# limited dynamic range

# If you want a more physical version

# A good next step is to split each compartment into:

# density

# elemental composition

# energy-dependent attenuation

In [19]:
import numpy as np
from dataclasses import dataclass
from typing import Dict, Tuple

from scipy.ndimage import gaussian_filter, rotate, distance_transform_edt

try:
    import xraylib
except ImportError:
    xraylib = None


In [20]:
# -----------------------------
# Labels
# -----------------------------
LABEL_BG = 0
LABEL_WATER = 0   # background is water
LABEL_CYTOPLASM = 1
LABEL_MEMBRANE = 2
LABEL_NUCLEUS = 3
LABEL_NUCLEOLUS = 4
LABEL_MITO = 5
LABEL_VESICLE = 6
LABEL_LIPID = 7


@dataclass
class Material:
    name: str
    density_g_cm3: float
    composition: Dict[str, float]   # mass fractions


# Hydrated compositions
COMPARTMENT_COMPOSITION = {
    "cytoplasm": {"H": 0.10, "C": 0.14, "N": 0.03, "O": 0.72, "P": 0.005, "S": 0.005},
    "membrane":  {"H": 0.10, "C": 0.68, "N": 0.02, "O": 0.18, "P": 0.015, "S": 0.005},
    "nucleus":   {"H": 0.10, "C": 0.18, "N": 0.05, "O": 0.63, "P": 0.03,  "S": 0.01},
    "protein": {"H": 0.065, "C": 0.53, "N": 0.16, "O": 0.23, "P": 0.00, "S": 0.00},
    "nucleicacid": {"H": 0.042, "C": 0.302, "N": 0.149, "O": 0.424, "P": 0.082, "S": 0.00},
}

# Reuse / slight extensions where needed
NUCLEOLUS_COMPOSITION = {"H": 0.10, "C": 0.20, "N": 0.06, "O": 0.58, "P": 0.045, "S": 0.015}
MITO_COMPOSITION      = {"H": 0.10, "C": 0.24, "N": 0.05, "O": 0.57, "P": 0.025, "S": 0.015}
VESICLE_COMPOSITION   = {"H": 0.1119, "O": 0.8881}  # water-like
LIPID_COMPOSITION     = {"H": 0.121, "C": 0.766, "N": 0.00, "O": 0.113, "P": 0.00}
WATER_COMPOSITION     = {"H": 0.1119, "O": 0.8881}

MATERIALS: Dict[int, Material] = {
    LABEL_WATER:     Material("water",         1.00, WATER_COMPOSITION),
    LABEL_CYTOPLASM: Material("cytoplasm",     1.03, COMPARTMENT_COMPOSITION["cytoplasm"]),
    LABEL_MEMBRANE:  Material("membrane",      1.10, COMPARTMENT_COMPOSITION["membrane"]),
    LABEL_NUCLEUS:   Material("nucleus",       1.06, COMPARTMENT_COMPOSITION["nucleus"]),
    LABEL_NUCLEOLUS: Material("nucleolus",     1.10, NUCLEOLUS_COMPOSITION),
    LABEL_MITO:      Material("mitochondrion", 1.08, MITO_COMPOSITION),
    LABEL_VESICLE:   Material("vesicle",       1.00, VESICLE_COMPOSITION),
    LABEL_LIPID:     Material("lipid_droplet", 0.92, LIPID_COMPOSITION),
}


In [21]:
ELEMENT_Z = {
    "H": 1,
    "C": 6,
    "N": 7,
    "O": 8,
    "P": 15,
    "S": 16,
}


def mass_attenuation_coefficient_from_composition(composition: Dict[str, float], energy_eV: float) -> float:
    """
    Return mass attenuation coefficient (mu/rho) in cm^2/g
    from elemental mass fractions using xraylib.
    """
    if xraylib is None:
        raise ImportError(
            "xraylib is required for composition-based mu calculation. "
            "Install with: pip install xraylib"
        )

    energy_keV = energy_eV / 1000.0
    mu_over_rho_cm2_g = 0.0

    total_w = sum(composition.values())
    if not np.isclose(total_w, 1.0, atol=1e-6):
        composition = {k: v / total_w for k, v in composition.items()}

    for elem, weight_fraction in composition.items():
        Z = ELEMENT_Z[elem]
        mu_over_rho_cm2_g += weight_fraction * xraylib.CS_Total(Z, energy_keV)

    return mu_over_rho_cm2_g


def linear_attenuation_um_inv(density_g_cm3: float, composition: Dict[str, float], energy_eV: float) -> float:
    """
    Convert density * mass attenuation into linear attenuation in 1/um.

    mu [cm^-1] = rho [g/cm^3] * (mu/rho) [cm^2/g]
    mu [um^-1] = mu [cm^-1] / 1e4
    """
    mu_over_rho = mass_attenuation_coefficient_from_composition(composition, energy_eV)
    mu_cm_inv = density_g_cm3 * mu_over_rho
    mu_um_inv = mu_cm_inv / 1.0e4
    return mu_um_inv


In [22]:
ENERGY_EV = 520.0

MU_520_TABLE = {
    label: linear_attenuation_um_inv(mat.density_g_cm3, mat.composition, ENERGY_EV)
    for label, mat in MATERIALS.items()
}


In [23]:
# -----------------------------
# Geometry helpers
# -----------------------------
def ellipsoid_mask(
    shape: Tuple[int, int, int],
    center: Tuple[float, float, float],
    radii: Tuple[float, float, float],
    rotation_deg_xyz: Tuple[float, float, float] = (0.0, 0.0, 0.0),
) -> np.ndarray:
    """
    Create a rotated ellipsoid mask in a (z, y, x) volume.
    Rotation order: x, y, z.
    """
    z, y, x = np.indices(shape)
    zz = z - center[0]
    yy = y - center[1]
    xx = x - center[2]

    pts = np.stack([xx.ravel(), yy.ravel(), zz.ravel()], axis=0)

    rx, ry, rz = np.deg2rad(rotation_deg_xyz)

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(rx), -np.sin(rx)],
        [0, np.sin(rx),  np.cos(rx)],
    ])
    Ry = np.array([
        [ np.cos(ry), 0, np.sin(ry)],
        [0, 1, 0],
        [-np.sin(ry), 0, np.cos(ry)],
    ])
    Rz = np.array([
        [np.cos(rz), -np.sin(rz), 0],
        [np.sin(rz),  np.cos(rz), 0],
        [0, 0, 1],
    ])

    R = Rz @ Ry @ Rx
    pts_rot = R.T @ pts

    x2 = pts_rot[0].reshape(shape)
    y2 = pts_rot[1].reshape(shape)
    z2 = pts_rot[2].reshape(shape)

    a, b, c = radii
    return (x2 / a) ** 2 + (y2 / b) ** 2 + (z2 / c) ** 2 <= 1.0


def random_point_in_ellipsoid(
    center: Tuple[float, float, float],
    radii: Tuple[float, float, float],
    rng: np.random.Generator,
    shrink: float = 1.0,
) -> Tuple[float, float, float]:
    """Sample a random point inside an ellipsoid."""
    while True:
        p = rng.uniform(-1.0, 1.0, size=3)
        if np.sum(p**2) <= 1.0:
            break

    scaled = p * np.array(radii) * shrink
    return (
        center[0] + scaled[2],
        center[1] + scaled[1],
        center[2] + scaled[0],
    )


def add_region(label_volume: np.ndarray, mask: np.ndarray, label: int, overwrite: bool = True) -> None:
    """Write a label into the label volume."""
    if overwrite:
        label_volume[mask] = label
    else:
        label_volume[(mask) & (label_volume == LABEL_BG)] = label


In [29]:
def build_cell_phantom(
    voxel_um: float = 0.015,
    shape: Tuple[int, int, int] = None,
    full_resolution: bool = False,
    n_mito: int = 18,
    n_vesicles: int = 35,
    n_lipid: int = 3,
    seed: int = 42,
) -> Dict[str, np.ndarray]:
    """
    Build a synthetic 3D cell phantom for soft X-ray absorption imaging at 520 eV.

    Parameters
    ----------
    shape : tuple
        Volume shape (z, y, x)
    voxel_um : float
        Voxel size in micrometers
    n_mito : int
        Number of mitochondria
    n_vesicles : int
        Number of watery vesicles
    n_lipid : int
        Number of lipid droplets
    seed : int
        RNG seed
    """
    rng = np.random.default_rng(seed)
# ---------- choose shape ----------
    if shape is None:
        if full_resolution:
            # Realistic physical volume: Z ~ 10 um, X/Y ~ 20 um (user requested)
            z_vox = int(round(10.0 / voxel_um))
            y_vox = int(round(20.0 / voxel_um))
            x_vox = int(round(20.0 / voxel_um))
            shape = (z_vox, y_vox, x_vox)
            print(f"[INFO] full_resolution=True -> shape={shape}  (~{shape[2]*voxel_um:.1f} µm x {shape[1]*voxel_um:.1f} µm x {shape[0]*voxel_um:.1f} µm)")
            print("WARNING: this array may be extremely large (memory/disk). Proceed only if you know your resources.")
        else:
            # Preview shape chosen to be runnable interactively in a notebook.
            # This keeps voxel_um at 0.015 µm but reduces FOV for prototyping.
            # Default preview: Z ~ 3 µm, X/Y ~ 6 µm => shape ~ (200, 400, 400)
            shape = (int(round(3.0 / voxel_um)), int(round(6.0 / voxel_um)), int(round(6.0 / voxel_um)))
            print(f"[INFO] full_resolution=False -> preview shape={shape}  (~{shape[2]*voxel_um:.2f} µm x {shape[1]*voxel_um:.2f} µm x {shape[0]*voxel_um:.2f} µm)")

    z_size_um = shape[0] * voxel_um
    y_size_um = shape[1] * voxel_um
    x_size_um = shape[2] * voxel_um

    # initialize with water background
    label_volume = np.full(shape, LABEL_WATER, dtype=np.uint8)

    # physical cell center in voxel coordinates
    cell_center = ((shape[0] - 1) / 2, (shape[1] - 1) / 2, (shape[2] - 1) / 2)

    # cell radii in voxels. Aim: ~20 µm XY, thinner in Z (if full_resolution True).
    # For preview mode, scale radii to fit the preview FOV.
    if full_resolution:
        cell_radii_um = (3.0, 9.0, 10.0)  # z-radius 3um -> thickness ~6um, y-radius 9um -> 18um diameter, x-radius 10um -> 20um
    else:
        # scale down to preview FOV proportionally to the shape
        # choose radii as fractions of the available FOV
        cell_radii_um = (min(3.0, z_size_um * 0.45), min(9.0, y_size_um * 0.45), min(10.0, x_size_um * 0.45))

    cell_radii = tuple(r_um / voxel_um for r_um in cell_radii_um)
    cell_mask = ellipsoid_mask(shape, cell_center, cell_radii, rotation_deg_xyz=(5, -3, 8))
    add_region(label_volume, cell_mask, LABEL_CYTOPLASM)

    # membrane shell (thin)

    # Distance-based membrane shell:
    # voxels inside the cell that are within membrane_thickness_um of the boundary
    membrane_thickness_um = 0.10  # 100 nm
    membrane_thickness_vox = membrane_thickness_um / voxel_um

    # distance to the nearest background voxel, measured inside the cell
    inside_dist = distance_transform_edt(cell_mask)

    membrane_mask = cell_mask & (inside_dist <= membrane_thickness_vox)
    cytoplasm_mask = cell_mask & (~membrane_mask)

    add_region(label_volume, cytoplasm_mask, LABEL_CYTOPLASM)
    add_region(label_volume, membrane_mask, LABEL_MEMBRANE)

    # nucleus (positioned slightly off-center)
    nucleus_center = (
        cell_center[0] - (0.3 / voxel_um),
        cell_center[1] + (0.8 / voxel_um),
        cell_center[2] - (1.0 / voxel_um),
    )
    nucleus_radii = (1.8 / voxel_um, 3.8 / voxel_um, 4.2 / voxel_um)
    nucleus_mask = ellipsoid_mask(shape, nucleus_center, nucleus_radii, rotation_deg_xyz=(0, 10, -18))
    nucleus_mask &= cytoplasm_mask
    add_region(label_volume, nucleus_mask, LABEL_NUCLEUS)

    # nucleolus
    nucleolus_center = (
        nucleus_center[0] + (0.2 / voxel_um),
        nucleus_center[1] - (0.5 / voxel_um),
        nucleus_center[2] + (0.4 / voxel_um),
    )
    nucleolus_radii = (0.7 / voxel_um, 1.0 / voxel_um, 1.0 / voxel_um)
    nucleolus_mask = ellipsoid_mask(shape, nucleolus_center, nucleolus_radii, rotation_deg_xyz=(0, 0, 22))
    nucleolus_mask &= nucleus_mask
    add_region(label_volume, nucleolus_mask, LABEL_NUCLEOLUS)

    # mitochondria (random elongated ellipsoids)
    for _ in range(n_mito):
        placed = False
        for _attempt in range(120):
            c = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.78)
            # avoid nucleus
            dz = (c[0] - nucleus_center[0]) / nucleus_radii[0]
            dy = (c[1] - nucleus_center[1]) / nucleus_radii[1]
            dx = (c[2] - nucleus_center[2]) / nucleus_radii[2]
            if dz**2 + dy**2 + dx**2 < 1.2:
                continue

            mito_radii = (
                rng.uniform(max(1.0, 3.0 / voxel_um), max(2.0, 6.0 / voxel_um)),
                rng.uniform(max(2.0, 4.0 / voxel_um), max(3.0, 7.0 / voxel_um)),
                rng.uniform(max(6.0, 10.0 / voxel_um), max(10.0, 18.0 / voxel_um)),
            )
            rot = (rng.uniform(-60, 60), rng.uniform(-60, 60), rng.uniform(-180, 180))
            mito_mask = ellipsoid_mask(shape, c, mito_radii, rot)
            mito_mask &= cytoplasm_mask
            mito_mask &= ~nucleus_mask
            if np.count_nonzero(mito_mask) < 20:
                continue
            add_region(label_volume, mito_mask, LABEL_MITO)
            placed = True
            break

    # vesicles
    for _ in range(n_vesicles):
        for _attempt in range(80):
            c = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.85)
            # choose radius in voxels (smaller for preview)
            r_min_um, r_max_um = (0.03, 0.12)  # 30 - 120 nm
            ves_r_um = rng.uniform(r_min_um, r_max_um)
            ves_r = ves_r_um / voxel_um
            ves_mask = ellipsoid_mask(shape, c, (ves_r, ves_r, ves_r))
            ves_mask &= cytoplasm_mask
            ves_mask &= ~nucleus_mask
            if np.count_nonzero(ves_mask) < 8:
                continue
            add_region(label_volume, ves_mask, LABEL_VESICLE)
            break

    # lipid droplets (high-contrast)
    for _ in range(n_lipid):
        for _attempt in range(80):
            c = random_point_in_ellipsoid(cell_center, cell_radii, rng, shrink=0.72)
            ld_r_um = rng.uniform(0.08, 0.2)  # 80 - 200 nm
            ld_r = ld_r_um / voxel_um
            ld_mask = ellipsoid_mask(shape, c, (ld_r, ld_r, ld_r))
            ld_mask &= cytoplasm_mask
            ld_mask &= ~nucleus_mask
            if np.count_nonzero(ld_mask) < 15:
                continue
            add_region(label_volume, ld_mask, LABEL_LIPID)
            break

 # -------------------------
    # Base density and mu maps
    # -------------------------
    density_volume = np.zeros(shape, dtype=np.float32)
    mu_volume = np.zeros(shape, dtype=np.float32)

    # Precompute MU table using xraylib (composition + density -> mu)
    if xraylib is None:
        raise ImportError("xraylib is required. Install in notebook: %pip install xraylib")

    MU_520_TABLE = {}
    for label, mat in MATERIALS.items():
        mu_val = linear_attenuation_um_inv(mat.density_g_cm3, mat.composition, ENERGY_EV)
        MU_520_TABLE[label] = mu_val

    for label, mat in MATERIALS.items():
        mask = label_volume == label
        density_volume[mask] = mat.density_g_cm3
        mu_volume[mask] = MU_520_TABLE[label]

    # -------------------------
    # Intra-compartment heterogeneity
    # -------------------------
    noise = rng.normal(0.0, 1.0, size=shape).astype(np.float32)
    noise = gaussian_filter(noise, sigma=3.0)
    noise = (noise - noise.mean()) / (noise.std() + 1e-8)

    cytoplasm_mask = label_volume == LABEL_CYTOPLASM
    nucleus_mask = label_volume == LABEL_NUCLEUS
    nucleolus_mask = label_volume == LABEL_NUCLEOLUS
    mito_mask = label_volume == LABEL_MITO
    membrane_mask = label_volume == LABEL_MEMBRANE
    vesicle_mask = label_volume == LABEL_VESICLE
    lipid_mask = label_volume == LABEL_LIPID
    water_mask = label_volume == LABEL_WATER

    # Density perturbations [g/cm^3] (small)
    density_volume[cytoplasm_mask] += 0.015 * noise[cytoplasm_mask]
    density_volume[nucleus_mask] += 0.012 * noise[nucleus_mask]
    density_volume[nucleolus_mask] += 0.008 * noise[nucleolus_mask]
    density_volume[mito_mask] += 0.010 * noise[mito_mask]
    density_volume[membrane_mask] += 0.006 * noise[membrane_mask]
    density_volume[vesicle_mask] += 0.003 * noise[vesicle_mask]
    density_volume[lipid_mask] += 0.004 * noise[lipid_mask]
    density_volume[water_mask] += 0.001 * noise[water_mask]

    # Clip to reasonable hydrated densities
    density_volume = np.clip(density_volume, 0.85, 1.4)

    # Recompute mu from perturbed density while keeping composition fixed
    for label, mat in MATERIALS.items():
        mask = label_volume == label
        mu_over_rho = mass_attenuation_coefficient_from_composition(mat.composition, ENERGY_EV)  # cm2/g
        mu_volume[mask] = density_volume[mask] * mu_over_rho / 1.0e4  # 1/um

    # small direct mu noise (composition-level microheterogeneity)
    mu_volume[cytoplasm_mask] += 0.015 * noise[cytoplasm_mask]
    mu_volume[nucleus_mask] += 0.020 * noise[nucleus_mask]
    mu_volume[mito_mask] += 0.020 * noise[mito_mask]
    mu_volume = np.clip(mu_volume, 0.0, None)

    return {
        "label_volume": label_volume,
        "density_volume": density_volume,
        "mu_volume": mu_volume,
        "voxel_um": float(voxel_um),
        "shape": shape,
        "fov_um": (z_size_um, y_size_um, x_size_um),
    }


In [30]:
def project_absorption(
    mu_volume: np.ndarray,
    voxel_um: float,
    axis: int = 0,
    I0: float = 1.0,
) -> np.ndarray:
    """
    Beer-Lambert projection for absorption imaging.

    I = I0 * exp(- integral(mu dl))

    mu_volume is in [1/um]
    voxel_um is in [um]
    """
    path_integral = np.sum(mu_volume, axis=axis) * voxel_um
    projection = I0 * np.exp(-path_integral)
    return projection.astype(np.float32)


def rotate_volume_for_angle(volume: np.ndarray, angle_deg: float) -> np.ndarray:
    """Rotate around z-axis for tomography-style acquisition."""
    return rotate(
        volume,
        angle=angle_deg,
        axes=(1, 2),
        reshape=False,
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=True,
    )


def simulate_sinogram(
    mu_volume: np.ndarray,
    voxel_um: float,
    angles_deg: np.ndarray,
) -> np.ndarray:
    """
    Simulate a simple parallel-beam projection stack.
    Output shape: (n_angles, y, x)
    """
    projections = []
    for angle in angles_deg:
        vol_rot = rotate_volume_for_angle(mu_volume, float(angle))
        proj = project_absorption(vol_rot, voxel_um=voxel_um, axis=0)
        projections.append(proj)
    return np.stack(projections, axis=0)

In [31]:
def summarize_volume(label_volume: np.ndarray, density_volume: np.ndarray, mu_volume: np.ndarray) -> None:
    print(f"Volume summary at {ENERGY_EV:.1f} eV")
    print("-" * 76)
    for label, material in MATERIALS.items():
        mask = label_volume == label
        n = np.count_nonzero(mask)
        if n == 0:
            continue
        mean_rho = float(density_volume[mask].mean())
        mean_mu = float(mu_volume[mask].mean())
        print(
            f"{label:2d}  {material.name:14s}  "
            f"voxels={n:10d}  density={mean_rho:.3f} g/cm^3  mu={mean_mu:.4f} 1/um"
        )


In [32]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox, fixed
from IPython.display import display


def normalize_limits(arr, pmin=1, pmax=99):
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        return 0.0, 1.0
    vmin, vmax = np.percentile(vals, [pmin, pmax])
    if vmin == vmax:
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


def get_slice(vol, axis, index):
    if axis == 0:
        return vol[index, :, :]
    elif axis == 1:
        return vol[:, index, :]
    elif axis == 2:
        return vol[:, :, index]
    else:
        raise ValueError("axis must be 0, 1, or 2")


def show_volume_slice(
    volume,
    labels=None,
    title="volume",
    cmap="gray",
    label_cmap="tab20",
):
    """
    Jupyter-friendly 3D slicer for numpy arrays.

    Parameters
    ----------
    volume : np.ndarray
        3D array to visualize
    labels : np.ndarray or None
        Optional 3D integer label array for overlay
    title : str
        Viewer title
    cmap : str
        Colormap for the main volume
    label_cmap : str
        Colormap for labels overlay
    """
    if volume.ndim != 3:
        raise ValueError(f"volume must be 3D, got shape {volume.shape}")

    if labels is not None:
        if labels.shape != volume.shape:
            raise ValueError("labels must have the same shape as volume")

    default_vmin, default_vmax = normalize_limits(volume)

    axis_widget = Dropdown(
        options=[("z", 0), ("y", 1), ("x", 2)],
        value=0,
        description="Axis:"
    )

    slice_widget = IntSlider(
        min=0,
        max=volume.shape[0] - 1,
        step=1,
        value=volume.shape[0] // 2,
        description="Slice:",
        continuous_update=False
    )

    vmin_widget = FloatSlider(
        min=float(np.nanmin(volume)),
        max=float(np.nanmax(volume)),
        step=(float(np.nanmax(volume)) - float(np.nanmin(volume)) + 1e-9) / 500.0,
        value=default_vmin,
        description="vmin:",
        continuous_update=False,
        readout_format=".4f",
        layout={"width": "90%"}
    )

    vmax_widget = FloatSlider(
        min=float(np.nanmin(volume)),
        max=float(np.nanmax(volume)),
        step=(float(np.nanmax(volume)) - float(np.nanmin(volume)) + 1e-9) / 500.0,
        value=default_vmax,
        description="vmax:",
        continuous_update=False,
        readout_format=".4f",
        layout={"width": "90%"}
    )

    overlay_widget = Checkbox(
        value=(labels is not None),
        description="Overlay labels",
        disabled=(labels is None)
    )

    alpha_widget = FloatSlider(
        min=0.0,
        max=1.0,
        step=0.05,
        value=0.35,
        description="Label α:",
        continuous_update=False
    )

    def update_slice_range(*args):
        ax = axis_widget.value
        slice_widget.max = volume.shape[ax] - 1
        if slice_widget.value > slice_widget.max:
            slice_widget.value = slice_widget.max

    axis_widget.observe(update_slice_range, names="value")

    def viewer(axis, slice_idx, vmin, vmax, overlay, alpha):
        img = get_slice(volume, axis, slice_idx)

        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(img, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")

        if labels is not None and overlay:
            lab = get_slice(labels, axis, slice_idx)
            masked_lab = np.ma.masked_where(lab == 0, lab)
            ax.imshow(masked_lab, cmap=label_cmap, alpha=alpha, origin="lower", interpolation="nearest")

        ax.set_title(f"{title} | axis={axis} | slice={slice_idx}")
        ax.set_xlabel("pixel")
        ax.set_ylabel("pixel")
        plt.colorbar(im, ax=ax, shrink=0.8)
        plt.show()

    interact(
        viewer,
        axis=axis_widget,
        slice_idx=slice_widget,
        vmin=vmin_widget,
        vmax=vmax_widget,
        overlay=overlay_widget,
        alpha=alpha_widget,
    )


In [33]:
# -----------------------------
# Example run (default preview)
# -----------------------------
if __name__ == "__main__" or True:
    # user settings
    voxel_um = 0.015  # per user request
    full_resolution = False  # set True if you want the large real-size grid (watch memory!)
    # Optional: you can pass an explicit shape e.g. shape=(667,1333,1333) but that will be huge.
    phantom = build_cell_phantom(voxel_um=0.05, full_resolution=full_resolution, shape=(200,200,200), seed=42)

    labels = phantom["label_volume"]
    density = phantom["density_volume"]
    mu = phantom["mu_volume"]
    voxel_um = phantom["voxel_um"]
    print(f"phantom shape = {phantom['shape']}, voxel = {voxel_um} µm, FOV (um z,y,x) = {phantom['fov_um']}")

    summarize_volume(labels, density, mu)

    # single projection (along z-axis)
    proj0 = project_absorption(mu, voxel_um=voxel_um, axis=0)
    print("projection shape:", proj0.shape, "min/max:", float(proj0.min()), float(proj0.max()))

    # small sinogram
    angles = np.linspace(0, 180, 16, endpoint=False)
    sino = simulate_sinogram(mu, voxel_um=voxel_um, angles_deg=angles)
    print("sinogram shape:", sino.shape)

    # save numpy outputs
    np.save("cell_labels.npy", labels)
    np.save("cell_density_gcm3.npy", density)
    np.save("cell_mu_520eV_uminv.npy", mu)
    np.save("cell_projection_0deg.npy", proj0)
    np.save("cell_sinogram.npy", sino)
    print("Saved: cell_labels.npy, cell_density_gcm3.npy, cell_mu_520eV_uminv.npy, cell_projection_0deg.npy, cell_sinogram.npy")

    # show interactive viewer (if running in notebook)
    try:
        show_volume_slice(mu, labels=labels, title="mu at 520 eV [1/um]", cmap="inferno")
    except Exception as e:
        print("[INFO] show_volume_slice failed:", e)
        # fallback: show a center slice
        import matplotlib.pyplot as plt
        ctr = mu.shape[0] // 2
        plt.imshow(mu[ctr], cmap="inferno", origin="lower")
        plt.title("mu center slice")
        plt.colorbar()
        plt.show()

phantom shape = (200, 200, 200), voxel = 0.05 µm, FOV (um z,y,x) = (10.0, 10.0, 10.0)
Volume summary at 520.0 eV
----------------------------------------------------------------------------
 0  water           voxels=   5964244  density=1.000 g/cm^3  mu=0.1178 1/um
 2  membrane        voxels=    137030  density=1.100 g/cm^3  mu=0.9997 1/um
 3  nucleus         voxels=    825469  density=1.061 g/cm^3  mu=0.4546 1/um
 4  nucleolus       voxels=     23480  density=1.101 g/cm^3  mu=0.5297 1/um
 5  mitochondrion   voxels=   1048432  density=1.080 g/cm^3  mu=0.5326 1/um
 6  vesicle         voxels=       923  density=1.001 g/cm^3  mu=0.1179 1/um
 7  lipid_droplet   voxels=       422  density=0.918 g/cm^3  mu=0.8738 1/um
projection shape: (200, 200) min/max: 0.00705143203958869 0.30829620361328125
sinogram shape: (16, 200, 200)
Saved: cell_labels.npy, cell_density_gcm3.npy, cell_mu_520eV_uminv.npy, cell_projection_0deg.npy, cell_sinogram.npy


interactive(children=(Dropdown(description='Axis:', options=(('z', 0), ('y', 1), ('x', 2)), value=0), IntSlide…

In [34]:
import tifffile as tiff
import numpy as np
import os

def save_tiff_stack(volume, filename, voxel_um=0.015):
    """
    Save a 3D numpy array as a TIFF stack (Z, Y, X).
    Includes voxel size metadata (in µm).
    """
    volume = np.asarray(volume)

    # Ensure correct dtype
    if volume.dtype == np.float64:
        volume = volume.astype(np.float32)

    # Metadata (ImageJ-compatible)
    metadata = {
        'spacing': voxel_um,   # z spacing
        'unit': 'um'
    }

    tiff.imwrite(
        filename,
        volume,
        imagej=True,
        metadata=metadata
    )
    print(f"Saved: {filename}  shape={volume.shape}  dtype={volume.dtype}")


# Example usage
save_tiff_stack(labels.astype(np.uint8), "cell_labels.tif", voxel_um)
save_tiff_stack(density.astype(np.float32), "cell_density.tif", voxel_um)
save_tiff_stack(mu.astype(np.float32), "cell_mu_520eV.tif", voxel_um)


Saved: cell_labels.tif  shape=(200, 200, 200)  dtype=uint8
Saved: cell_density.tif  shape=(200, 200, 200)  dtype=float32
Saved: cell_mu_520eV.tif  shape=(200, 200, 200)  dtype=float32


In [13]:
show_volume_slice(density, labels=labels, title="Density [g/cm^3]", cmap="viridis")


interactive(children=(Dropdown(description='Axis:', options=(('z', 0), ('y', 1), ('x', 2)), value=0), IntSlide…

In [ ]:
show_volume_slice(mu, labels=labels, title="mu at 520 eV [1/um]", cmap="inferno")
